In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from numba import njit

In [2]:
df = pd.read_csv("RAW DATA/BTC_1sec.csv")

We now estimate the parameters of the hawkes process, using L-BFGS-B.

In [3]:
df.head()

,Unnamed: 0,system_time,midpoint,spread,buys,sells,bids_distance_0,bids_distance_1,bids_distance_2,bids_distance_3,...,asks_market_notional_5,asks_market_notional_6,asks_market_notional_7,asks_market_notional_8,asks_market_notional_9,asks_market_notional_10,asks_market_notional_11,asks_market_notional_12,asks_market_notional_13,asks_market_notional_14
0,0,2021-04-07 11:32:42.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,2021-04-07 11:32:43.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,2021-04-07 11:32:44.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,2021-04-07 11:32:45.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,2021-04-07 11:32:46.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
times = np.array(df.index)

In [5]:
length = len(times)

In [ ]:
# Refer to "Volumes.ipynb". 
X = np.genfromtxt("volumes_and_mask",delimiter=",")

In [7]:
volumes = np.array(X[:,0])
mask = np.asarray(X[:,1],int)

We cut the data for the sake of numerical stability.

In [26]:
mask_cut = mask[0:101]

In [27]:
times_cut = times[0:101]
volumes_cut = volumes[0:101]

In [28]:
length = times_cut[-1]

In [29]:
arrival_times_cut = times_cut[mask_cut==1]
volumes_arrival_cut = volumes_cut[mask_cut==1]

In [30]:
print((volumes_arrival_cut))

[ 194099.169952  479674.695351  651812.708542  681032.79213
 1255953.193443  890004.664032  706337.782848  819758.803116
  887103.766031  905154.641453  712580.916611  358494.145449
  258861.12678   604496.508121  468336.441771  167588.2104
 2525465.587379  598838.097519 3466840.123474 3066015.420708
 3054836.211415 3129236.671616 3235410.538818 3247074.00145
 2894136.033435 2368269.310257  564914.045689 2544846.572304
 1172449.390381  975406.359657  796810.498333  469564.551014
 1131851.272205 1111705.052258 1322359.669281  601931.496124
  650624.018723  551325.152245  502111.582001  475668.328857
  542651.169416  809699.640211  736574.124691  437069.223265
  681248.013332  577958.847858  431928.240231  592207.175632
  572938.341625  517682.613262  570587.539412  604697.732256
  530202.367905  732306.790495  690749.27964   668170.41198
  837002.749006  509602.042217  261105.318321  282422.224121
  215735.790489  697426.053719]


We write the negative likelihood function, and a generic optimization code. 


In [34]:
@njit
def compute_nll_identity_marks(params, times, marks):
    """
    #MLE for Hawkes with phi(kappa) = kappa (Identity).
    #params: [mu, a, b]
    """
    mu,a, b = params
    
    # Non-negativity constraints
    if mu <= 0  or a<0 or b <= 0:
        return 1e15

    n = len(times)
    log_intensity_sum = 0.0
    R = 0.0
    
    for i in range(n):
        if i == 0:
            intensity = mu
        else:
            dt = times[i] - times[i-1]
            # Recursive update using the mark of the PREVIOUS event
            # R_i = exp(-b*dt) * (R_{i-1} + kappa_{i-1})
            R = np.exp(-b * dt) * (R + marks[i-1])
            intensity = mu +  a * R
        
        log_intensity_sum += np.log(max(intensity, 1e-10))

    # The Compensator Integral (Λ) over [0, T]
    T_max = length
    compensator = mu * T_max
    
    for i in range(n):
        # Integral of excitation: (a * kappa_i / b) * (1 - exp(-b(T_max - T_i)))
        compensator += ( a * marks[i] / b) * (1 - np.exp(-b * (T_max - times[i])))

    return compensator - log_intensity_sum

def optimize_hawkes_identity(times, marks):
    # Scale times for numerical stability if they are in millions
    scale_factor = 1
    times_scaled = times / scale_factor
    
    # Initial guess [mu, a, b]
    # mu is scaled up because time units are now 1000x larger
    avg_rate = (len(times) / times_scaled[-1])
    initial_guess = [avg_rate*0.5 ,0.1,1.5]

    
    
    bounds = [(1e-7, None),(0,None), (1e-7, None)]
    
    epsilon = 1e-7
    
    

    result = minimize(
        compute_nll_identity_marks, 
        initial_guess, 
        args=(times_scaled, marks), 
        bounds=bounds, 
        method='L-BFGS-B'
    )
    
    

    if result.success:
        
        optimized_nll = result.fun
        
        
        
        
        
        mu_opt, a_opt, b_opt = result.x
        # Rescale parameters back to original 'seconds' units
        return {
            "mu": mu_opt / scale_factor,
             "a": a_opt,# 'a' is a multiplier, it stays the same
            "b": b_opt / scale_factor,
            "nll": optimized_nll,
        }
    else:
        raise ValueError(result.message)

In [35]:
res = optimize_hawkes_identity(arrival_times_cut,np.log((volumes_arrival_cut)))

In [36]:
print(res)

{'mu': 0.16509275456073078, 'a': 0.011478042431186214, 'b': 0.20471162648542252, 'nll': 86.09190010781765}


To see that our model indeed satisfies the stationary condition/burst factor. We want $\hat{h}<1$, and in our context of a gaussian distributed marks with an exponential excitation kernel, we have $\hat{h}= \frac{a \mu}{b}$.

Again, for more information, refer to the paper.

In [37]:
print(res["a"]/res["b"]*13.32)

0.7468433904231008
